In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

embeddings = OpenAIEmbeddings(
    model=os.environ["EMBEDDING_MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"], # type: ignore
    check_embedding_ctx_length=False,
)

loader = TextLoader("../documents/data.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
docs = text_splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=docs, embedding=embeddings)
vector_store.save_local(folder_path="../db/python/data")

# FAISS.from_texts

vector_store.similarity_search(query="在线客服", k=2)

[Document(id='ace844d9-0efd-44ea-b733-4e721aab3450', metadata={'source': '../documents/data.txt'}, page_content='三、虚拟助手与实时互动\n应用场景：在线客服、虚拟导购、智能教育助手。\n技术实现：WebRTC提供实时音视频交互能力，AI通过语音识别和自然语言处理理解用户需求，并提供智能回复或引导。'),
 Document(id='35c48059-9d41-472d-8850-4b6923b446e3', metadata={'source': '../documents/data.txt'}, page_content='案例：电商平台使用虚拟导购助手，通过WebRTC与用户实时互动，推荐商品并解答问题。')]

In [6]:
from langchain_community.vectorstores import FAISS
from pprint import pprint

vector_store = FAISS.load_local(
    folder_path="../db/python/data",
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
)

retriever = vector_store.as_retriever(search_kwargs={'k': 2})

rst = retriever.invoke('在线客服')

# rst = vector_store.similarity_search(query="在线客服", k=2)

pprint(rst)

[Document(id='f2bf8ee5-ec7f-477a-bf46-8a6e7255b3d7', metadata={'source': '../documents/data.txt'}, page_content='三、虚拟助手与实时互动\n应用场景：在线客服、虚拟导购、智能教育助手。\n技术实现：WebRTC提供实时音视频交互能力，AI通过语音识别和自然语言处理理解用户需求，并提供智能回复或引导。'),
 Document(id='5b2f8740-8600-4dd7-b35f-0de3a1d8db2f', metadata={'source': '../documents/data.txt'}, page_content='案例：电商平台使用虚拟导购助手，通过WebRTC与用户实时互动，推荐商品并解答问题。')]


In [ ]:
from langchain_postgres import PGVectorStore, PGEngine
import os

engine = PGEngine.from_connection_string(url=os.environ['POSTGRES_URL'])

table_name = os.environ['POSTGRES_TABLE_NAME']

try:
    engine.init_vectorstore_table(table_name=table_name, vector_size=1536)
except Exception as e:
    print(e)
    pass

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import os

embeddings = OpenAIEmbeddings(
    model=os.environ['EMBEDDING_MODEL_NAME'],
    base_url=os.environ['BASE_URL'],
    api_key=os.environ['OPENAI_API_KEY'],
    check_embedding_ctx_length=False,
)

print(len(embeddings.embed_query("你好")))

loader = TextLoader('../documents/data.txt')
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)
docs = text_splitter.split_documents(documents)

table_name = os.environ['POSTGRES_TABLE_NAME']

vector_store = PGVectorStore.from_documents(
    engine=engine,
    documents=docs,
    embedding=embeddings,
    table_name=table_name
)


In [ ]:
engine.drop_table(table_name=table_name)

In [ ]:
store = PGVectorStore.create_sync(engine=engine, embedding_service=embeddings, table_name='langchain_test')
store.similarity_search(query="客服", k=2)